In [ ]:
df=df = pd.read_csv('processed_data.csv')

# Ensure 'num_pages' is numeric and handle potential NaNs for page-based calculations
df['num_pages'] = pd.to_numeric(df['num_pages'], errors='coerce')

# Create a new DataFrame for user-level aggregates
user_metrics = pd.DataFrame(df['user_id'].unique(), columns=['user_id'])
user_metrics = user_metrics.set_index('user_id')

# Group by user_id for calculations
user_grouped = df.groupby('user_id')

# 1. Calculate overall_shelf_to_read_conversion
# Count of books where converted_within_target is 1
converted_count = user_grouped['converted_within_target'].apply(lambda x: (x == 1).sum())
# Total books added by that user
total_books_added = user_grouped.size()

user_metrics['overall_shelf_to_read_conversion'] = converted_count / total_books_added

# Helper function to calculate conversion rate for a given page range
def calculate_page_conversion(group_df, min_pages, max_pages=None):
    # Filter books with valid num_pages within the range
    if max_pages is None:
        # For 'long' category (num_pages > max_pages_threshold)
        filtered_books = group_df[group_df['num_pages'] > min_pages]
    else:
        # For 'short' and 'medium' categories
        filtered_books = group_df[
            (group_df['num_pages'] >= min_pages) & 
            (group_df['num_pages'] < max_pages)
        ]
    
    total_in_category = len(filtered_books)
    if total_in_category == 0:
        return 0.0 # Avoid division by zero
    
    completed_in_category = (filtered_books['converted_within_target'] == 1).sum()
    return completed_in_category / total_in_category


# 2. Calculate short_conversion_rate (num_pages < 200)
# Use df.apply and lambda to pass the filtered group_df to the helper function
user_metrics['short_conversion_rate'] = user_grouped.apply(
    lambda x: calculate_page_conversion(x.dropna(subset=['num_pages']), 0, 200)
)

# 3. Calculate medium_conversion_rate (num_pages >= 200 to 400)
user_metrics['medium_conversion_rate'] = user_grouped.apply(
    lambda x: calculate_page_conversion(x.dropna(subset=['num_pages']), 200, 400)
)

# 4. Calculate long_conversion_rate (num_pages > 400)
user_metrics['long_conversion_rate'] = user_grouped.apply(
    lambda x: calculate_page_conversion(x.dropna(subset=['num_pages']), 400)
)


# Reset index to make user_id a column again and display results
user_metrics = user_metrics.reset_index()
print("\nUser-level Conversion Metrics:")
display(user_metrics.head())

print("\nSummary statistics for conversion rates:")
display(user_metrics.describe())

In [ ]:
# Optional: Save the new user_metrics DataFrame to a CSV file
output_filename_metrics = 'user_conversion_metrics.csv'
user_metrics.to_csv(output_filename_metrics, index=False)
print(f"\nUser conversion metrics saved to '{output_filename_metrics}'")